In [ ]:
import sys
import os
import gc
from pathlib import Path  # noqa: F401

# Resolve project root regardless of where the notebook is launched from
for _candidate in [".", "..", "../.."]:
    _p = os.path.abspath(_candidate)
    if os.path.isdir(os.path.join(_p, "src")):
        sys.path.insert(0, _p)
        break

import numpy as np  # noqa: E402
import pandas as pd  # noqa: E402
import matplotlib.pyplot as plt  # noqa: E402
import seaborn as sns  # noqa: E402
import mne  # noqa: E402
from mne.time_frequency import tfr_array_morlet  # noqa: E402

from src.preprocessing.pipeline import DatasetHandler  # noqa: E402
from src.definitions.fields import (  # noqa: E402
    ExperimentNames,
    CoordinateSystems,
    PreprocessedDataVariants,
    SingleDataMetadata,
    ConditionVariants,
    MusicTypeVariants,
)
from src.definitions.constants import AssrEpoch, ProjectPaths  # noqa: E402
from src.preprocessing.stimulus_alignment import (  # noqa: E402
    DEFAULT_STIMULUS_LABEL,
    StimulusAligner,
    get_stimulus_onset_samples,
)

%matplotlib inline
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.0)
mne.set_log_level("ERROR")
print("Setup complete.")

# ASSR Stimulus-Onset Offset — Where Is `fam+` Relative to the Sound?

The ASSR analyses assume the `fam+` annotation marks the **onset** of the 500 ms
40 Hz train (that is what `AssrEpoch` encodes: baseline before 0, stimulus in
`[0, 0.5]` s). Time-frequency maps of the aligned data contradict this: the driven
40 Hz activity sits mostly at **negative** times, with `t = 0` falling near the
*end* of the stimulus.

This notebook measures where the stimulus actually is, and checks whether the
shift is introduced anywhere in our own preprocessing / alignment.

**Method.** The read-out is 40 Hz **inter-trial phase coherence (ITC)** locked to
the `fam+` markers. ITC is the right instrument here: the ASSR is phase-locked to
the stimulus, so ITC is elevated only while the train is driving the cortex, and
its rise/fall bracket the stimulus in time. A 4-cycle Morlet kernel at 40 Hz is
100 ms long, i.e. ~±50 ms of temporal smearing — short enough to resolve a 500 ms
train, unlike a narrow FIR band-pass whose impulse response is several hundred ms.
The kernel is symmetric, so it blurs but never shifts.

**Steps.**
1. Ground truth in the *untouched* EDF (no processing at all, `first_samp == 0`).
2. Per-subject stimulus window and onset offset on continuous `RAW_AFTER_ICA`.
3. Group-average ITC profile with a 2-D (lag × duration) box fit.
4. Stage-by-stage comparison — does any pipeline step move the data?
5. Verification that the `first_samp` bookkeeping is applied exactly once.
6. Aligned time axes: wavelet cache vs concatenated array.

**Result obtained when this notebook was written** (19 Placebo ASSR recordings):
the driven response spans **[-370, +100] ms** relative to `fam+` with a fitted
width of **470 ms** (paradigm: 500 ms), i.e. the acoustic onset precedes the
marker by **≈380 ms** (per-subject median). The same shift is already present in
the raw EDF, so it is a property of the source annotations, not of this pipeline.

## Configuration

In [ ]:
# ── Group under test ─────────────────────────────────────────────────────────
EXPERIMENT = ExperimentNames.ASSR
CONDITION = ConditionVariants.PLACEBO
MUSIC_TYPE = MusicTypeVariants.ASSR
STIMULUS_LABEL = DEFAULT_STIMULUS_LABEL  # "fam+"

# ── ITC measurement ──────────────────────────────────────────────────────────
ITC_FREQ = 40.0  # ASSR driving frequency
ITC_N_CYCLES = 4.0  # 4 cycles @ 40 Hz = 100 ms kernel -> ~±50 ms smearing
EPOCH_PRE_S = 1.0  # epoch window around each marker: symmetric, so the stimulus
EPOCH_POST_S = 1.0  # fits under either hypothesis (before / after the marker),
# and short enough that the neighbouring stimulus (shortest inter-onset interval
# ~1.26 s, response ending ~1.16 s before the next marker) stays outside it.
EDGE_GUARD_S = 2 * ITC_N_CYCLES / ITC_FREQ  # 200 ms: a Morlet transform is not
# trustworthy within ~2 kernel lengths of an epoch edge, so peaks and baselines
# are only ever read from the interior.

# Channel subsampling — the offset is a timing question, not a topography one,
# so a spatial subsample keeps every step fast without changing the answer.
CHANNEL_STRIDE = 7
N_CHANNELS = 30

# Recordings to use. None = the whole group (recommended: the group fit in step 3
# needs the pooled SNR).
N_SUBJECTS = None

# ── Box-fit search grids ─────────────────────────────────────────────────────
# The lag grid is deliberately SYMMETRIC about 0: a grid weighted towards negative
# lags would presuppose the very conclusion this notebook is testing.
LAG_GRID_S = np.arange(-0.45, 0.455, 0.005)  # candidate acoustic-onset lags
DUR_GRID_S = np.arange(0.30, 0.71, 0.01)  # candidate train durations

# ── Later-stage checks ───────────────────────────────────────────────────────
RESAMPLE_FREQ = 250.0  # rate of the concatenated / wavelet data
KEEP_TAIL_S = 0.1  # StimulusAligner default, re-used when replanning
CONCAT_LABEL = f"{CONDITION.value}_{MUSIC_TYPE.value}"

# ── Plot saving ──────────────────────────────────────────────────────────────
SAVE_PLOTS = True
PLOTS_DIR = (
    ProjectPaths.NOTEBOOKS_DIR
    / "00-preprocessing"
    / "plots"
    / "assr_stimulus_onset_offset"
)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Group        : {CONDITION.value} / {MUSIC_TYPE.value} ({EXPERIMENT.value})")
print(f"Marker label : {STIMULUS_LABEL!r}")
print(
    f"ITC          : {ITC_FREQ} Hz, {ITC_N_CYCLES:.0f} cycles "
    f"({ITC_N_CYCLES / ITC_FREQ * 1000:.0f} ms kernel)"
)
print(
    f"Paradigm     : {AssrEpoch.STIMULUS_DURATION_S * 1000:.0f} ms train, "
    f"AssrEpoch assumes it occupies [0, {AssrEpoch.STIMULUS_DURATION_S}] s"
)
print(f"Plots -> {PLOTS_DIR}")

## Helper Functions

`itc_profile` is the measurement; `half_max_window` and `fit_boxcar` are two
independent ways of reading the stimulus window off a profile (a peak-anchored
threshold crossing and a box fit), so the conclusion does not rest on one
estimator.

Both are guarded against the two ways this measurement can fool you: a Morlet
transform spikes at the epoch edges (hence `EDGE_GUARD_S`, and a peak searched
only in the interior), and a search range skewed towards negative lags would
build the expected answer into the estimate (hence the symmetric `LAG_GRID_S`).
`fit_boxcar` is the primary estimator — it uses the whole profile shape and
degrades gracefully at low SNR, whereas a half-max crossing depends on a single
peak sample.

In [ ]:
def itc_profile(
    data,
    onsets,
    sfreq,
    pre_s=EPOCH_PRE_S,
    post_s=EPOCH_POST_S,
    freq=ITC_FREQ,
    n_cycles=ITC_N_CYCLES,
):
    """Inter-trial phase coherence at `freq`, locked to `onsets`.

    :param data: ``(n_channels, n_times)`` array.
    :param onsets: Onset sample indices *into this array*.
    :param sfreq: Sampling frequency of ``data`` in Hz.
    :return: ``(times, itc, n_epochs)`` with times in seconds, 0 at the marker.
    """
    pre, post = int(round(pre_s * sfreq)), int(round(post_s * sfreq))
    keep = [int(o) for o in onsets if o - pre >= 0 and o + post < data.shape[-1]]
    epochs = np.stack(
        [np.asarray(data[:, o - pre : o + post], dtype=float) for o in keep]
    )
    tfr = tfr_array_morlet(
        epochs, sfreq, [freq], n_cycles=n_cycles, output="complex", verbose=False
    )
    # Collapse any singleton frequency/taper axes, keep (n_epochs, n_ch, n_times).
    z = tfr.reshape(tfr.shape[0], tfr.shape[1], -1, tfr.shape[-1])[:, :, 0, :]
    z = z / (np.abs(z) + 1e-30)  # unit modulus: phase only
    itc = np.abs(z.mean(axis=0)).mean(axis=0)  # over trials, then channels
    return np.arange(-pre, post) / sfreq, itc, len(keep)


def half_max_window(times, itc, baseline_end_s=-0.5, edge_guard_s=EDGE_GUARD_S):
    """Peak-anchored half-maximum window of an ITC profile.

    The peak is searched only inside the edge-guarded interior, and the returned
    crossings are the ones immediately bracketing *that* peak. Both matter: the
    Morlet transform produces spurious spikes at the epoch edges, and taking the
    first/last crossing in the whole profile lets any such spike invert the
    window. For a symmetric epoch the search interval is symmetric about 0, so
    this estimator does not favour negative times.

    :param baseline_end_s: Interior samples earlier than this form the
        stimulus-free baseline.
    :param edge_guard_s: Margin excluded at each epoch edge.
    :return: ``(rise, fall, snr)`` — edge times in seconds and peak/baseline.
    :raises ValueError: If the guarded baseline window is empty.
    """
    interior = (times >= times[0] + edge_guard_s) & (times <= times[-1] - edge_guard_s)
    baseline = interior & (times < baseline_end_s)
    if not baseline.any():
        raise ValueError(
            "Empty baseline window — widen the epoch or raise baseline_end_s."
        )
    base = itc[baseline].mean()
    interior_idx = np.flatnonzero(interior)
    peak = interior_idx[np.argmax(itc[interior])]
    half = base + 0.5 * (itc[peak] - base)

    rise_idx = peak
    while rise_idx > interior_idx[0] and itc[rise_idx - 1] >= half:
        rise_idx -= 1
    fall_idx = peak
    while fall_idx < interior_idx[-1] and itc[fall_idx + 1] >= half:
        fall_idx += 1
    return times[rise_idx], times[fall_idx], itc[peak] / base


def fit_boxcar(times, itc, lag_grid=LAG_GRID_S, dur_grid=DUR_GRID_S):
    """Grid-search the box maximising mean(inside) - mean(outside) ITC.

    :param lag_grid: Candidate acoustic-onset lags relative to the marker (s).
    :param dur_grid: Candidate train durations (s). Pass a single value to fit
        the lag only, with the duration fixed by the paradigm.
    :return: ``(lag, duration)`` in seconds.
    """
    y = itc - itc.mean()
    dt = times[1] - times[0]
    best = (-np.inf, np.nan, np.nan)
    for dur in np.atleast_1d(dur_grid):
        n = int(round(dur / dt))
        for lag in lag_grid:
            i0 = int(round((lag - times[0]) / dt))
            if i0 < 0 or i0 + n >= len(y):
                continue
            score = y[i0 : i0 + n].mean() - np.concatenate([y[:i0], y[i0 + n :]]).mean()
            if score > best[0]:
                best = (score, lag, dur)
    return best[1], best[2]


def load_picked(
    handler, filename, variant, stride=CHANNEL_STRIDE, n_channels=N_CHANNELS
):
    """Load a processed recording with only a channel subsample in memory."""
    raw = handler.load_data_file(
        filename, is_processed=True, processed_data_type=variant, preload=False
    )
    raw.pick(raw.ch_names[::stride][:n_channels]).load_data()
    return raw


def ms(value):
    """Format a duration in seconds as a signed millisecond string."""
    return "  n/a  " if np.isnan(value) else f"{value * 1000:+7.1f} ms"

## Dataset & Group Selection

In [ ]:
dataset_handler = DatasetHandler(
    EXPERIMENT, CoordinateSystems.HYDROGEL_257_NO_FIDUCIALS
)
metadata = dataset_handler.dataset_metadata
group_df = metadata[
    (metadata[SingleDataMetadata.CONDITION] == CONDITION)
    & (metadata[SingleDataMetadata.MUSIC_TYPE] == MUSIC_TYPE)
]
group_files = group_df[SingleDataMetadata.FILENAME].tolist()
if N_SUBJECTS is not None:
    group_files = group_files[:N_SUBJECTS]

print(
    f"{EXPERIMENT.value}: {len(metadata)} recordings, "
    f"{len(group_df)} in {CONDITION.value}/{MUSIC_TYPE.value}, "
    f"{len(group_files)} used here"
)
group_df.head()

## Step 1 — Ground Truth in the Untouched EDF

Nothing in this step touches our pipeline: the EDF is read straight from
`data/raw/`, so `first_samp == 0` and the `fam+` annotation onsets *are* sample
indices. If the shift shows up here, it cannot have been introduced by
preprocessing.

The only processing applied is an average reference, which is spatial and cannot
move anything in time.

A single recording carries little ITC, so read the **box fit** here; the half-max
window is printed for completeness but on one recording it is dominated by noise.

In [ ]:
REFERENCE_FILE = group_files[0]
edf_path = dataset_handler.raw_data_dir / REFERENCE_FILE

raw_edf = mne.io.read_raw_edf(edf_path, preload=True).pick("eeg")
raw_edf.set_eeg_reference("average", verbose=False)
sfreq_edf = raw_edf.info["sfreq"]

assert raw_edf.first_samp == 0, "untouched EDF must start at sample 0"
marker_seconds = np.array(
    [
        onset
        for onset, desc in zip(
            raw_edf.annotations.onset, raw_edf.annotations.description
        )
        if desc.strip().lower() == STIMULUS_LABEL
    ]
)
onsets_edf = np.round(marker_seconds * sfreq_edf).astype(int)

# Every marker label present, to confirm which one we are locking to.
labels, counts = np.unique(raw_edf.annotations.description, return_counts=True)
print(f"{REFERENCE_FILE}")
print(f"  annotation labels : {dict(zip(labels, counts))}")
print(
    f"  {STIMULUS_LABEL!r} markers : {len(onsets_edf)}, "
    f"median inter-onset interval {np.median(np.diff(marker_seconds)):.3f} s"
)

data_edf = (
    raw_edf.copy().pick(raw_edf.ch_names[::CHANNEL_STRIDE][:N_CHANNELS]).get_data()
)
times_edf, itc_edf, n_edf = itc_profile(data_edf, onsets_edf, sfreq_edf)
rise_edf, fall_edf, snr_edf = half_max_window(times_edf, itc_edf)
lag_edf, _ = fit_boxcar(times_edf, itc_edf, dur_grid=[AssrEpoch.STIMULUS_DURATION_S])

del raw_edf, data_edf
gc.collect()

print(f"\n  40 Hz ITC over {n_edf} epochs, peak/baseline = x{snr_edf:.2f}")
print(f"  half-max window : [{ms(rise_edf)}, {ms(fall_edf)}]")
print(
    f"  best onset lag (duration fixed at "
    f"{AssrEpoch.STIMULUS_DURATION_S * 1000:.0f} ms) : {ms(lag_edf)}"
)
print(
    "\n  No pipeline code has touched this recording: the shift measured here "
    "cannot\n  have been introduced by preprocessing."
)

## Step 2 — Per-Subject Window on Continuous `RAW_AFTER_ICA`

`RAW_AFTER_ICA` is the cleanest **continuous** stage: it has been coarse-cropped
(a single crop, no splicing), filtered and ICA-cleaned, so it has good SNR *and*
an intact time axis — no splice seams that could fake a response edge.

Onsets come from `get_stimulus_onset_samples`, the same helper the pipeline uses,
so this measures exactly what the pipeline sees.

Read the table with `itc_snr` in mind: single-recording ITC is noisy, and in the
weakest recordings the half-max columns describe a noise excursion rather than
the response (their `width_ms` then departs badly from the paradigm's 500 ms).
`onset_lag_ms`, the box fit, is the column to judge — its spread across
recordings is the evidence that the offset is a fixed property of the paradigm
rather than a per-subject accident.

In [ ]:
rows = []
profiles = []
for filename in group_files:
    raw = load_picked(dataset_handler, filename, PreprocessedDataVariants.RAW_AFTER_ICA)
    sfreq = raw.info["sfreq"]
    onsets = get_stimulus_onset_samples(raw, STIMULUS_LABEL)
    times, itc, n_epochs = itc_profile(raw.get_data(), onsets, sfreq)
    rise, fall, snr = half_max_window(times, itc)
    lag, _ = fit_boxcar(times, itc, dur_grid=[AssrEpoch.STIMULUS_DURATION_S])
    rows.append(
        {
            "recording": filename[:10],
            "n_epochs": n_epochs,
            "itc_snr": round(snr, 2),
            "rise_ms": round(rise * 1000, 1),
            "fall_ms": round(fall * 1000, 1),
            "width_ms": round((fall - rise) * 1000, 1),
            "onset_lag_ms": round(lag * 1000, 1),
        }
    )
    profiles.append(itc)
    itc_times = times
    del raw
    gc.collect()

per_subject = pd.DataFrame(rows)
lags = per_subject["onset_lag_ms"]
grid_bound_ms = LAG_GRID_S.max() * 1000
pinned = int(lags.abs().ge(grid_bound_ms - 1e-6).sum())

print(
    f"per-subject onset lag: median {lags.median():+.1f} ms, "
    f"IQR [{lags.quantile(0.25):+.1f}, {lags.quantile(0.75):+.1f}] ms"
)
print(
    f"  within ±50 ms of the median : "
    f"{int((lags - lags.median()).abs().le(50).sum())}/{len(lags)}"
)
print(
    f"  pinned at the ±{grid_bound_ms:.0f} ms search bound "
    f"(low SNR, no usable fit) : {pinned}/{len(lags)}"
)
print(
    "  -> quote the median: with some fits sitting on the search bound, mean "
    "and sd\n     are not meaningful summaries of this column."
)
per_subject

## Step 3 — Group Profile and 2-D Box Fit

Pooling the per-subject profiles raises SNR enough to fit the onset lag **and**
the duration simultaneously. The fitted duration is the sanity check: if it lands
near the paradigm's 500 ms, the box is tracking the real train and not a noise
excursion.

In [ ]:
group_itc = np.stack(profiles).mean(axis=0)
group_rise, group_fall, group_snr = half_max_window(itc_times, group_itc)
fit_lag, fit_dur = fit_boxcar(itc_times, group_itc)

print(f"group ITC (n={len(profiles)} recordings): peak/baseline = x{group_snr:.2f}")
print(
    f"  half-max window     : [{ms(group_rise)}, {ms(group_fall)}]  "
    f"(width {(group_fall - group_rise) * 1000:.0f} ms)"
)
print(f"  best-fit box        : onset {ms(fit_lag)}, duration {fit_dur * 1000:.0f} ms")
print(f"  paradigm train      : {AssrEpoch.STIMULUS_DURATION_S * 1000:.0f} ms")
print(
    f"  => stimulus spans [{fit_lag * 1000:+.0f}, {(fit_lag + fit_dur) * 1000:+.0f}] ms "
    f"relative to the {STIMULUS_LABEL!r} marker"
)

fig, ax = plt.subplots(figsize=(11, 4.5))
for prof in profiles:
    ax.plot(itc_times * 1000, prof, color="0.8", lw=0.7, zorder=1)
ax.plot(
    itc_times * 1000, group_itc, color="C0", lw=2.2, label="group mean ITC", zorder=3
)
ax.axvspan(
    fit_lag * 1000,
    (fit_lag + fit_dur) * 1000,
    color="C1",
    alpha=0.18,
    zorder=0,
    label=f"fitted stimulus [{fit_lag * 1000:+.0f}, {(fit_lag + fit_dur) * 1000:+.0f}] ms",
)
ax.axvspan(
    0,
    AssrEpoch.STIMULUS_DURATION_S * 1000,
    facecolor="none",
    edgecolor="C3",
    hatch="//",
    lw=1.2,
    zorder=2,
    label="window AssrEpoch assumes",
)
ax.axvline(0, color="k", lw=1.4, ls="--", zorder=4, label=f"{STIMULUS_LABEL!r} marker")
ax.set_xlabel(f"time relative to {STIMULUS_LABEL!r} (ms)")
ax.set_ylabel(f"{ITC_FREQ:.0f} Hz inter-trial phase coherence")
ax.set_title(
    f"ASSR 40 Hz ITC locked to {STIMULUS_LABEL!r} — {CONDITION.value}, "
    f"n={len(profiles)} recordings"
)
ax.legend(loc="upper left", fontsize=9)
plt.tight_layout()
if SAVE_PLOTS:
    fig.savefig(PLOTS_DIR / "itc_group_profile.png", dpi=150)
plt.show()

## Step 4 — Does Any Pipeline Stage Move the Data?

The same measurement is repeated at each stage that follows `RAW_AFTER_ICA`:

| stage | data | onsets used |
|---|---|---|
| `RAW_AFTER_ICA` | 1000 Hz, continuous | `fam+` annotations |
| `RAW_CROPPED` | 1000 Hz, spliced + cross-subject aligned | `fam+` annotations |
| `CONCATENATED` | 250 Hz stacked array | saved `.stimulus_onsets.npy` |

The diagnostic is the **trailing** edge: it is the one edge the alignment does not
touch. If it stays put across stages, no stage shifts the data relative to the
markers. The leading edge is expected to move, because the aligner keeps only
`keep_tail_sec` of continuous data before each onset and splices the rest away —
truncation, not translation.

In [ ]:
STAGE_PRE_S, STAGE_POST_S = 0.9, 0.8  # the spliced stages have no genuine data
# earlier than -keep_tail_sec, so their pre-onset stretch acts as baseline

stage_rows = []
skipped = []

for variant, label in [
    (PreprocessedDataVariants.RAW_AFTER_ICA, "after_ica (continuous)"),
    (PreprocessedDataVariants.RAW_CROPPED, "cropped (spliced+aligned)"),
]:
    stage_profiles, stage_times, stage_sfreq = [], None, None
    for filename in group_files:
        try:
            raw = load_picked(dataset_handler, filename, variant)
        except FileNotFoundError:
            skipped.append(f"{label}: {filename}")
            continue
        stage_sfreq = raw.info["sfreq"]
        stage_times, itc, _ = itc_profile(
            raw.get_data(),
            get_stimulus_onset_samples(raw, STIMULUS_LABEL),
            stage_sfreq,
            pre_s=STAGE_PRE_S,
            post_s=STAGE_POST_S,
        )
        stage_profiles.append(itc)
        del raw
        gc.collect()
    rise, fall, snr = half_max_window(
        stage_times, np.stack(stage_profiles).mean(axis=0)
    )
    stage_rows.append(
        {
            "stage": label,
            "sfreq": stage_sfreq,
            "n_recordings": len(stage_profiles),
            "itc_snr": round(snr, 2),
            "rise_ms": round(rise * 1000, 1),
            "fall_ms": round(fall * 1000, 1),
        }
    )

# Concatenated array + the onsets that were saved next to it.
concat_dir = (
    ProjectPaths.PROCESSED_DATA_DIR
    / EXPERIMENT.value
    / PreprocessedDataVariants.CONCATENATED.value
)
concat = np.load(concat_dir / f"{CONCAT_LABEL}.npy", mmap_mode="r")
saved_onsets = np.load(
    concat_dir / f"{CONCAT_LABEL}{ProjectPaths.STIMULUS_ONSETS_SUFFIX}"
)
concat_profiles = []
for subject in range(concat.shape[0]):
    picked = np.asarray(concat[subject][::CHANNEL_STRIDE][:N_CHANNELS])
    concat_times, itc, _ = itc_profile(
        picked, saved_onsets, RESAMPLE_FREQ, pre_s=STAGE_PRE_S, post_s=STAGE_POST_S
    )
    concat_profiles.append(itc)
rise, fall, snr = half_max_window(concat_times, np.stack(concat_profiles).mean(axis=0))
stage_rows.append(
    {
        "stage": "concatenated (saved onsets)",
        "sfreq": RESAMPLE_FREQ,
        "n_recordings": concat.shape[0],
        "itc_snr": round(snr, 2),
        "rise_ms": round(rise * 1000, 1),
        "fall_ms": round(fall * 1000, 1),
    }
)

if skipped:
    print(f"skipped {len(skipped)} missing file(s):")
    for item in skipped:
        print(f"  {item}")
stages = pd.DataFrame(stage_rows)
print("\ntrailing edge across stages (should be constant):", stages["fall_ms"].tolist())
stages

## Step 5 — The `first_samp` Bookkeeping Is Applied Exactly Once

The coarse crop removes the recording lead-in but keeps MNE's absolute sample
numbering, so a cropped recording has `first_samp > 0` while its annotation
onsets stay on the original clock. `get_stimulus_onset_samples` maps between the
two by subtracting `raw.first_time` — once. The asserts below fail loudly if that
correction is missing, doubled, or lost across resampling and saving.

In [ ]:
raw = load_picked(
    dataset_handler, REFERENCE_FILE, PreprocessedDataVariants.RAW_AFTER_ICA
)
sfreq = raw.info["sfreq"]
onsets = get_stimulus_onset_samples(raw, STIMULUS_LABEL)
absolute = np.round(
    np.array(
        [
            onset
            for onset, desc in zip(raw.annotations.onset, raw.annotations.description)
            if desc.strip().lower() == STIMULUS_LABEL
        ]
    )
    * sfreq
).astype(int)

print(f"{REFERENCE_FILE} ({PreprocessedDataVariants.RAW_AFTER_ICA.value})")
print(
    f"  first_samp = {raw.first_samp}, first_time = {raw.first_time:.3f} s, "
    f"n_times = {raw.n_times}"
)
print(f"  first marker: absolute sample {absolute[0]}, data-relative {onsets[0]}")

# 1. the correction is exactly `first_samp`, applied once
assert np.array_equal(onsets, np.sort(absolute) - raw.first_samp)
# 2. onsets index inside the cropped array (they would not if it were skipped)
assert onsets[-1] < raw.n_times
print("  OK  onsets == absolute - first_samp, and all inside the array")

# 3. it survives resampling: first_samp and onsets scale by the same ratio
resampled = raw.copy().resample(RESAMPLE_FREQ, n_jobs=1)
onsets_resampled = get_stimulus_onset_samples(resampled, STIMULUS_LABEL)
ratio = RESAMPLE_FREQ / sfreq
print(
    f"  after resample -> {RESAMPLE_FREQ:.0f} Hz: first_samp "
    f"{raw.first_samp} -> {resampled.first_samp} (expected "
    f"{round(raw.first_samp * ratio)})"
)
assert np.abs(onsets_resampled - onsets * ratio).max() <= 1
print(f"  OK  onsets match onsets*{ratio} to within 1 sample")

del raw, resampled
gc.collect()

# 4. the saved onsets equal the aligned (spliced) onsets, rescaled
cropped = dataset_handler.load_data_file(
    REFERENCE_FILE,
    is_processed=True,
    processed_data_type=PreprocessedDataVariants.RAW_CROPPED,
    preload=False,
)
cropped_onsets = get_stimulus_onset_samples(cropped, STIMULUS_LABEL)
rescaled = np.round(cropped_onsets * RESAMPLE_FREQ / cropped.info["sfreq"]).astype(int)
print(
    f"\n  cropped onsets (1000 Hz) -> /{cropped.info['sfreq'] / RESAMPLE_FREQ:.0f}: "
    f"{rescaled[:5]}"
)
print(f"  saved .stimulus_onsets.npy               : {saved_onsets[:5]}")
print(f"  max |difference| = {np.abs(rescaled - saved_onsets).max()} samples")
del cropped
_ = gc.collect()

## Step 6 — Aligned Time Axes: Wavelet Cache vs Concatenated Array

Two independent code paths produce an "aligned" time axis:

* the **time-domain** path splices at 1000 Hz (`RAW_CROPPED`) and resamples afterwards;
* the **wavelet** path (`EEGSummarizedAnalyzer.load_pre_alignment_data`) resamples
  to 250 Hz *first* and then re-fits `StimulusAligner` on that grid, so wavelets are
  computed on continuous data and trimmed afterwards.

Both plans are rebuilt here from the group's onsets and compared against the saved
onsets and the two on-disk array lengths. Rounding happens per interval, so any
disagreement accumulates along the recording — which is why the comparison looks
at the *last* stimulus, not just the first.

In [ ]:
onsets_1000, onsets_250, lengths_1000, lengths_250 = [], [], [], []
concat_meta = pd.read_csv(concat_dir / f"{CONCAT_LABEL}.metadata.csv").sort_values(
    str(SingleDataMetadata.CONCATENATED_PERSON_INDEX)
)
concat_files = concat_meta[str(SingleDataMetadata.FILENAME)].tolist()

for filename in concat_files:
    raw = dataset_handler.load_data_file(
        filename,
        is_processed=True,
        processed_data_type=PreprocessedDataVariants.RAW_AFTER_ICA,
        preload=False,
    )
    markers = np.array(
        [
            onset
            for onset, desc in zip(raw.annotations.onset, raw.annotations.description)
            if desc.strip().lower() == STIMULUS_LABEL
        ]
    )
    sfreq = raw.info["sfreq"]
    ratio = RESAMPLE_FREQ / sfreq
    # first_time on each grid: resampling rescales first_samp, then rounds.
    first_time_1000 = raw.first_samp / sfreq
    first_time_250 = int(round(raw.first_samp * ratio)) / RESAMPLE_FREQ
    onsets_1000.append(
        np.sort(np.round((markers - first_time_1000) * sfreq).astype(int))
    )
    onsets_250.append(
        np.sort(np.round((markers - first_time_250) * RESAMPLE_FREQ).astype(int))
    )
    lengths_1000.append(raw.n_times)
    lengths_250.append(int(round(raw.n_times * ratio)))
    del raw

plan_1000 = StimulusAligner(onsets_1000, lengths_1000, sfreq, keep_tail_sec=KEEP_TAIL_S)
plan_250 = StimulusAligner(
    onsets_250, lengths_250, RESAMPLE_FREQ, keep_tail_sec=KEEP_TAIL_S
)

wavelet_dir = (
    ProjectPaths.PROCESSED_DATA_DIR / EXPERIMENT.value / "wavelets" / "broadband"
)
wavelet_files = sorted(wavelet_dir.glob(f"{CONCAT_LABEL}*.npz"))
wavelet_axis = None
if wavelet_files:
    import zipfile
    import numpy.lib.format as npformat

    with zipfile.ZipFile(wavelet_files[0]) as archive:
        with archive.open("data.npy") as handle:
            major, minor = npformat.read_magic(handle)
            reader = (
                npformat.read_array_header_1_0
                if (major, minor) == (1, 0)
                else npformat.read_array_header_2_0
            )
            wavelet_shape, _, _ = reader(handle)
    wavelet_axis = wavelet_shape[-1]
    print(
        f"wavelet cache : {wavelet_files[0].name}\n                shape={wavelet_shape}"
    )
else:
    print(f"no wavelet cache found under {wavelet_dir}")

print("\naligned axis lengths")
print(
    f"  plan @{sfreq:.0f} Hz total_length = {plan_1000.total_length}"
    f"  (/{sfreq / RESAMPLE_FREQ:.0f} = {plan_1000.total_length / (sfreq / RESAMPLE_FREQ):.2f})"
)
print(f"  plan @{RESAMPLE_FREQ:.0f} Hz total_length = {plan_250.total_length}")
print(f"  concatenated array on disk  = {concat.shape[-1]}")
print(f"  wavelet cache on disk       = {wavelet_axis}")

scaled_1000 = np.round(plan_1000.aligned_onset_samples * RESAMPLE_FREQ / sfreq).astype(
    int
)
drift = pd.DataFrame(
    {
        "stimulus": np.arange(len(saved_onsets)),
        "saved": saved_onsets,
        f"plan@{sfreq:.0f}Hz_rescaled": scaled_1000,
        f"plan@{RESAMPLE_FREQ:.0f}Hz": plan_250.aligned_onset_samples,
    }
)
drift["diff_1000_vs_saved"] = drift[f"plan@{sfreq:.0f}Hz_rescaled"] - drift["saved"]
drift["diff_250_vs_saved"] = drift[f"plan@{RESAMPLE_FREQ:.0f}Hz"] - drift["saved"]
print(
    f"\nmax |plan@{sfreq:.0f} rescaled - saved| = "
    f"{drift['diff_1000_vs_saved'].abs().max()} samples"
)
print(
    f"max |plan@{RESAMPLE_FREQ:.0f} - saved|      = "
    f"{drift['diff_250_vs_saved'].abs().max()} samples "
    f"({drift['diff_250_vs_saved'].abs().max() / RESAMPLE_FREQ * 1000:.0f} ms)"
)
drift.iloc[[0, 1, 2, len(drift) // 2, -3, -2, -1]]

## Conclusions

1. **The marker is late by ≈380 ms.** The 40 Hz driven response occupies roughly
   `[-370, +100]` ms around `fam+`, with a fitted width close to the paradigm's
   500 ms train. `t = 0` therefore falls in the last ~100 ms of the stimulus.
   `bgin` is 2 ms from `fam+`, so it is the same event, not an alternative onset
   marker.

2. **Our pipeline does not cause it.** The shift is already there in the untouched
   EDF (step 1), and the trailing edge of the response is unchanged across
   `after_ica` → `cropped` → `concatenated` (step 4). The `first_samp`
   bookkeeping is applied exactly once and survives resampling and saving
   (step 5). The offset is a property of the source annotations, so the intended
   value should be confirmed against the paradigm documentation rather than taken
   from this fit alone.

3. **Two consequences to fix.**
   * `StimulusAligner(keep_tail_sec=0.1)` keeps only 100 ms before each onset and
     splices the middle of each interval away — with the true response at
     `[-370, +100]` ms, most of the driven interval is discarded, and every onset
     is preceded by a splice seam at a fixed −100 ms latency (a phase-locked
     discontinuity in every trial).
   * `AssrEpoch` encodes stimulus `[0, 0.5]` s, so `AssrEpoch.stimulus_mask` — used
     by the ASSR PCA scripts and the IVA quality references — currently selects
     post-stimulus silence as the driven interval.

4. **Where a correction belongs.** `get_stimulus_onset_samples` is the single point
   through which every consumer reads onsets (coarse crop, aligner, analyzers), so
   a per-experiment marker→onset offset applied there propagates everywhere.
   Applying it requires re-running alignment, concatenation and the wavelet cache.

5. **Independent of the offset:** the wavelet cache and the concatenated array sit
   on two separately-planned aligned axes whose onsets drift apart along the
   recording (step 6). One path should be authoritative for the aligned axis.